# How loss trains attention: routing, content, and saturation

Chapter 4 · Session 2. Companion to sections 4.5–4.8 of the [chapter](../../book/chapters/04-attention-and-the-causal-information-boundary.md).

Follow the forward notebook first or read its solutions. This session uses a tiny fixed next-token classification head, one observed target, and no optimizer steps. You may read the complete solutions now and return to the optional coding exercises later. Tests verify the reference path; they do not assess your understanding.

In [ ]:
from pathlib import Path
import sys
import math
import torch
import torch.nn.functional as F
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/dongxi_llms/attention_evidence.py').exists())
if str(root / 'src') not in sys.path: sys.path.insert(0, str(root / 'src'))
from dongxi_llms.causal_attention_lab import attention_trace, teaching_inputs
from dongxi_llms.attention_evidence import gradient_evidence, scaling_evidence, cache_fixture, full_stack, decode_step, cache_evidence
torch.set_printoptions(precision=5, sci_mode=False)
print('PyTorch', torch.__version__, '| CPU float64')

## 1. Prediction: does loss supervise the attention map?

A single cross-entropy target is attached to position 2 (zero-based). Does loss update only V, or Q and K as well? Can earlier input rows receive gradients even though they have no direct target? Can row 3?

In [ ]:
prediction_gradient_paths = ""

### Reference solution

Q/K determine routing and V determines transmitted content; both routes can affect the prediction. Earlier input rows can receive credit through the chosen position's keys and values. Row 3 cannot affect position 2, so its input gradient is zero for this isolated loss. Shared parameters still update; that does not mean row 3 supplied information.

Run the following transparent forward/backward pass. The class label supplies no target attention matrix.

In [ ]:
x, wq, wk, wv = [t.clone().requires_grad_() for t in teaching_inputs()]
q, k, v = x @ wq, x @ wk, x @ wv
trace = attention_trace(q, k, v)
a, o = trace['weights'], trace['output']
for t in (q, k, v, a, o, trace['scaled']): t.retain_grad()
head = x.new_tensor([[1., -.5, .2], [-.3, .7, 1.]])
logits = o[2] @ head
loss = F.cross_entropy(logits[None], torch.tensor([1]))
loss.backward()
print('loss:', loss.item(), '| logits:', logits.detach())
print('input gradients by position:\n', x.grad)
assert x.grad[3].count_nonzero() == 0
assert x.grad[:2].norm() > 0

## 2. Implementation: derive the backward pass

Let G_O be the incoming derivative. Fill the expressions for G_V, G_A, and G_R. All tensors use rows for positions; R is the masked scaled score matrix. Treat the mask as fixed. Then extend the derivation through Q/K and the projection matrices.

In [ ]:
def my_backward(a, v, go):
    gv = ...
    ga = ...
    gr = ...
    return gv, ga, gr
run_my_backward = False
if run_my_backward:
    mine = my_backward(a.detach(), v.detach(), o.grad)

### Reference solution

G_V=AᵀG_O and G_A=G_OVᵀ. Each row's score gradient is A multiplied elementwise by its incoming derivative minus its A-weighted average. The subtraction reflects competition within a row.

Next, G_Q=G_R K/√d_k and G_K=G_RᵀQ/√d_k; projection gradients are Xᵀ times the projected-vector gradient. X receives all three paths added together. The fixed forbidden entries have zero score gradient.

In [ ]:
go = o.grad
manual_gv = a.detach().T @ go
manual_ga = go @ v.detach().T
manual_gr = a.detach() * (manual_ga - (manual_ga * a.detach()).sum(-1, keepdim=True))
for expected, actual in [(manual_gv, v.grad), (manual_ga, a.grad), (manual_gr, trace['scaled'].grad)]:
    torch.testing.assert_close(expected, actual, atol=1e-12, rtol=1e-12)
if run_my_backward:
    for actual, expected in zip(mine, [manual_gv, manual_ga, manual_gr]):
        torch.testing.assert_close(actual, expected)
report = gradient_evidence()
print('Manual versus autograd errors:', report['manual_errors'])
print('Central-difference max error:', report['finite_difference_error'])
assert max(report['manual_errors'].values()) < 1e-12
assert report['finite_difference_error'] < 1e-7
assert report['forbidden_score_gradient'] == 0

## 3. Intervention: freeze a route without changing the forward answer

Predict what `a.detach() @ v` and `a @ v.detach()` do. Does detaching change the loss value? Which projection parameters lose their gradient path?

In [ ]:
prediction_detach = ""

### Reference solution

Detach preserves the tensor's forward value and removes its backward connection. Detaching A prevents gradients reaching W_Q and W_K in this isolated graph; W_V still receives gradients. Detaching V removes W_V's path while routing can still learn from fixed values. These statements assume independent projection parameters and no other loss paths.

In [ ]:
for choice in [None, 'routing', 'values']:
    r = gradient_evidence(choice)
    print('Detach:', choice, '| loss:', r['loss'], '| gradient norms:', r['gradient_norms'])

## 4. Prediction and experiment: why divide by √d_k?

Under independent zero-mean unit-variance coordinates, Var(q·k)=d_k. Without scaling, wider heads create larger score gaps even before learning. Predict how this affects entropy and local sensitivity.

Entropy is −Σ a log a. The softmax Jacobian's trace is 1−Σ a². Near a one-hot row this trace approaches zero. This is a sensitivity measure, not a gradient of a language-model loss.

In [ ]:
prediction_scaling = ""

### Reference solution

Unscaled score spread grows approximately as √d_k under these assumptions. Scaling controls that spread and usually avoids near-one-hot initial rows. The exact statistics vary with the random draws. Learned queries/keys need not satisfy the IID assumptions.

This does not contradict Chapter 3's strong p−q correction for confidently wrong vocabulary predictions. That result uses the special combination of softmax and cross-entropy. Attention receives a derivative through a value mixture, so the same cancellation is not generally present.

In [ ]:
rows = scaling_evidence()
for row in rows:
    print({key: round(value, 5) if isinstance(value, float) else value for key, value in row.items()})
# Optional: change widths, seed, rows, or keys in scaling_evidence(...).
# Compare paired scaled/unscaled scores at each width, not unrelated draws.

## 5. Interpretation: can different attention maps yield the same result?

If the third value is the average of the first two, must changing attention weights change the output? Predict before running.

In [ ]:
prediction_explanation = ""

### Reference solution

Different distributions can transport the same value mixture. Therefore attention weights alone do not even uniquely determine the head's contribution without V, much less explain the final model prediction.

In [ ]:
values = torch.tensor([[2., 0.], [0., 2.], [1., 1.]], dtype=torch.float64)
weights_a = torch.tensor([.4, .4, .2], dtype=values.dtype)
weights_b = torch.tensor([.1, .1, .8], dtype=values.dtype)
print(weights_a @ values, weights_b @ values)
torch.testing.assert_close(weights_a @ values, weights_b @ values)

## Evidence boundary

Write what the verified derivatives and interventions establish.

In [ ]:
my_evidence_boundary = ""

### Reference solution

Analytical derivatives agree with autograd and finite differences for a fixed fixture. Controlled detach operations isolate two paths. An IID simulation illustrates the scaling argument. None of these establishes semantic roles for heads, trained-model capability, or GPU speed. The complete derivation and exercise answers are in the chapter and its [solutions](../../book/solutions/04-attention-and-the-causal-information-boundary.md).